In [0]:
%sql
SELECT current_catalog();

In [0]:
%sql
USE CATALOG harsha_databricks;

In [0]:
%sql
SELECT current_catalog();

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS harsha_databricks.bronze;

CREATE SCHEMA IF NOT EXISTS harsha_databricks.silver;

CREATE SCHEMA IF NOT EXISTS harsha_databricks.gold;

In [0]:
display(spark.sql("SHOW SCHEMAS"))

In [0]:
display(spark.sql("SHOW VOLUMES IN harsha_databricks.default"))

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS harsha_databricks.default.landing;

In [0]:
display(spark.sql("SHOW VOLUMES IN harsha_databricks.default"))

In [0]:
display(dbutils.fs.ls("/Volumes/harsha_databricks/default/landing/"))

In [0]:
customers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/harsha_databricks/default/landing/customers_sales.csv")
)

display(customers_df)

In [0]:
customers_df.printSchema()

In [0]:
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("harsha_databricks.bronze.customers2")

In [0]:
%sql show external locations

In [0]:
customers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/harsha_databricks/default/landing/customers_sales.csv")
)

display(customers_df)

In [0]:
%sql
SELECT *
FROM harsha_databricks.bronze.customers;

In [0]:
%sql DESCRIBE TABLE harsha_databricks.bronze.customers

In [0]:
bronze_df = spark.sql("""
    SELECT *
    FROM harsha_databricks.bronze.customers
""")

display(bronze_df)

In [0]:
from pyspark.sql.functions import trim, upper, col

silver_df = (
    bronze_df
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("email", trim(col("email")))
    .withColumn("country", upper(trim(col("country"))))
    .withColumn("status", upper(trim(col("status"))))
    .filter(col("purchase_amount") > 0)
)

display(silver_df)

In [0]:
%sql
SELECT *
FROM harsha_databricks.bronze.customers;

In [0]:
bronze_df = spark.sql("""
    SELECT *
    FROM harsha_databricks.bronze.customers
""")

display(bronze_df)

In [0]:
from pyspark.sql.functions import trim, upper, col

silver_df = (
    bronze_df
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("email", trim(col("email")))
    .withColumn("country", upper(trim(col("country"))))
    .withColumn("status", upper(trim(col("status"))))
    .filter(col("purchase_amount") > 0)
)

display(silver_df)

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("harsha_databricks.silver.customers")

In [0]:
%sql
SELECT *
FROM harsha_databricks.silver.customers;

In [0]:
%sql
SHOW TABLES IN harsha_databricks.silver;

In [0]:
%sql
SELECT
    product_purchased,
    COUNT(*) AS total_orders,
    ROUND(SUM(purchase_amount), 2) AS total_sales,
    ROUND(AVG(purchase_amount), 2) AS average_order_value
FROM harsha_databricks.silver.customers
GROUP BY product_purchased
ORDER BY total_sales DESC;

In [0]:
gold_df = spark.sql("""
    SELECT
        product_purchased,
        COUNT(*) AS total_orders,
        ROUND(SUM(purchase_amount), 2) AS total_sales,
        ROUND(AVG(purchase_amount), 2) AS average_order_value
    FROM harsha_databricks.silver.customers
    GROUP BY product_purchased
    ORDER BY total_sales DESC
""")

display(gold_df)

In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("harsha_databricks.gold.product_sales_summary")

In [0]:
%sql
SELECT *
FROM harsha_databricks.gold.product_sales_summary;

In [0]:
%sql
SHOW TABLES IN harsha_databricks.gold;

In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT customer_id) AS unique_customers,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_ids,
    SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) AS null_emails,
    SUM(CASE WHEN purchase_amount <= 0 THEN 1 ELSE 0 END) AS invalid_amounts
FROM harsha_databricks.silver.customers;